# XGBoost Model Building

**Project:** N₂O Emissions in Sub-Saharan Africa  
**Notebook objective:** Development, validation, tuning, diagnosis, and export of an XGBoost regression model for predicting N₂O flux.

This notebook documents the transition from exploratory data analysis to supervised model building. The modeling setup is based on the main observations from the EDA:

- The target variable is continuous, strongly right-skewed, and affected by extreme values.
- The dataset contains repeated observations that are grouped by `Event`.
- `Event` is treated as a grouping variable for validation and is excluded from the predictor set to reduce data leakage.
- Spatial coordinates, land-use information, meteorological variables, soil variables, fertilizer-related variables, and engineered temporal features are used as predictors.
- Model performance is evaluated on previously unseen events using a grouped train-test split.

# Table of Contents 
1. [ Modeling Strategy](#1-modeling-strategy)
2. [Imports and Configuration](#2-imports-and-configuration)
3. [Data Loading](#3-data-loading)
4. [Target and Feature Configuration](#4-target-and-feature-configuration)
5. [Feature Engineering](#5-feature-engineering)
6. [Train-Test Split by Event](#6-train-test-split-by-event)
7. [Preprocessing Pipeline](#7-preprocessing-pipeline)
8. [Baseline Model](#8-baseline-model)
9. [Initial XGBoost Model](#9-initial-xgboost-model)
10. [Hyperparameter Tuning with Grouped Cross-Validation](#10-hyperparameter-tuning-with-grouped-cross-validation)
11. [Final Model Evaluation](#11-final-model-evaluation)
12. [Error Analysis](#12-error-analysis)
13. [Feature Importance](#13-feature-importance)
14. [Diagnostic Plots](#14-diagnostic-plots)
15. [Save Model Artifacts](#15-save-model-artifacts)
16. [Optional Final Refit on All Data](#16-optional-final-refit-on-all-data)
17. [Summary and Next Steps](#17-summary-and-next-steps)

## 1. Modeling Strategy

The objective of the modeling step is to estimate the modeled N₂O flux from environmental, spatial, land-use, soil, and fertilizer-related predictors.

The modeling design follows four main decisions:

1. **Grouped validation:** Observations from the same `Event` are likely not independent. A random row-level split could therefore lead to overly optimistic performance estimates. For this reason, `Event` is used as the grouping variable for the holdout split and for cross-validation.
2. **Target transformation:** The EDA showed that N₂O flux is skewed and includes negative values. A shifted `log1p` transformation is tested for the tuned model. The shift is derived only from the training target to prevent leakage from the test set.
3. **Categorical encoding:** `Land use` is encoded with one-hot encoding. `Event` is not used as a direct model feature, because it mainly identifies measurement groups and could cause leakage.
4. **Robust evaluation:** Model quality is assessed using MAE, RMSE, R², median absolute error, bias, and residual summaries. MAE is especially relevant because the target contains extreme emission values.

## 2. Imports and Configuration

This section defines the required libraries, global plotting settings, warning behavior, and reproducibility settings used throughout the modeling workflow.

In [ ]:
from __future__ import annotations

%load_ext autoreload
%autoreload 2

import sys
from pathlib import Path
from typing import Any, Dict, Iterable, List, Optional, Sequence, Tuple

import warnings

import joblib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.base import clone
from sklearn.compose import ColumnTransformer
from sklearn.dummy import DummyRegressor
from sklearn.impute import SimpleImputer
from sklearn.metrics import (
    mean_absolute_error,
    mean_squared_error,
    median_absolute_error,
    r2_score,
)
from sklearn.model_selection import GroupKFold, GroupShuffleSplit, RandomizedSearchCV, cross_validate
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder

from xgboost import XGBRegressor


project_root = Path.cwd().parent
sys.path.append(str(project_root))

# from src.n2o_ml import io # import find_existing_path
from src.n2o_ml.io import find_existing_path, load_dataset,validate_required_columns
from src.n2o_ml.model_selection import split_by_group, build_group_kfold, summarize_split_distribution
from src.n2o_ml.preprocessing import make_one_hot_encoder, infer_feature_types, build_tabular_preprocessor, get_feature_type_lists
from src.n2o_ml.evaluation import regression_metrics, metrics_by_group
from src.n2o_ml.preprocessing import get_feature_type_lists
from src.n2o_ml.target_transform import ShiftedLogTargetTransformer
from src.n2o_ml.feature_engineering import add_n2o_domain_features



sns.set_theme(style="whitegrid", context="notebook")
warnings.filterwarnings("ignore", category=UserWarning)

RANDOM_STATE = 42
TEST_SIZE = 0.20
N_ITER_SEARCH = 30
N_JOBS = -1


## 3. Data Loading

The dataset is loaded from a set of possible project paths. This makes the notebook executable from different working directories, such as the repository root, the `notebooks/` folder, or a temporary execution environment.

In [ ]:
project_root = Path.cwd().parent if Path.cwd().name.lower() == "notebooks" else Path.cwd()

DATA_PATH = find_existing_path(
    [
        project_root / "data" / "interim" / "n2o_ssa_landuse_aligned_combined.csv",
        project_root / "data" / "processed" / "n2o_ssa_landuse_aligned_combined.csv",
        Path("../data/interim/n2o_ssa_landuse_aligned_combined.csv"),
        Path("../data/processed/n2o_ssa_landuse_aligned_combined.csv"),
        Path("/mnt/data/n2o_ssa_landuse_aligned_combined.csv"),
    ],
    verbose=True,
)

df_raw = load_dataset(DATA_PATH, verbose=True)
df_raw.head()


## 4. Target and Feature Configuration

This section defines the target variable, grouping column, date column, and required columns. The configuration is kept explicit so that later preprocessing and validation steps remain reproducible and easy to inspect.

In [ ]:
TARGET_COLUMN = "N2O flux [µg/m**2/h] (From soil surface, Modeled)"
GROUP_COLUMN = "Event"
DATE_COLUMN = "Date/Time"
LAND_USE_COLUMN = "Land use"

REQUIRED_COLUMNS = [
    TARGET_COLUMN,
    GROUP_COLUMN,
    DATE_COLUMN,
    LAND_USE_COLUMN,
    "Latitude",
    "Longitude",
]


validate_required_columns(df_raw, REQUIRED_COLUMNS, verbose=True)

print(f"Target column: {TARGET_COLUMN}")
print(f"Group column : {GROUP_COLUMN}")
print(f"Date column  : {DATE_COLUMN}")
print(f"Land use     : {LAND_USE_COLUMN}")


In [ ]:
# Quick target overview before modeling.
df_raw[TARGET_COLUMN].describe()


## 5. Feature Engineering

The feature engineering step converts the date column and creates additional temporal, cyclic, and interaction-based predictors. These features are derived from the EDA results and from process-related assumptions about N₂O emissions.

The raw date column is not used directly as a model feature. Instead, structured temporal information such as year, month, seasonality, and cyclic encodings is extracted.

In [ ]:
df_model = add_n2o_domain_features(df_raw, date_column=DATE_COLUMN, verbose=True)
df_model.head()

In [ ]:
# Define columns that must not be used as model input features.
drop_columns = [
    TARGET_COLUMN,
    DATE_COLUMN,
    GROUP_COLUMN,
]

# Check whether all required columns exist in the modeling dataframe.
missing_columns = [
    column for column in drop_columns
    if column not in df_model.columns
]

if missing_columns:
    raise ValueError(f"Missing required columns: {missing_columns}")

# Check whether the target contains missing values before training.
if df_model[TARGET_COLUMN].isna().any():
    raise ValueError(
        "Target column contains missing values. Handle them before modeling."
    )

# Create feature matrix, target vector, and group vector.
X = df_model.drop(columns=drop_columns)
y = df_model[TARGET_COLUMN].copy()
groups = df_model[GROUP_COLUMN].copy()

print(f"Feature matrix shape: {X.shape}")
print(f"Target shape        : {y.shape}")
print(f"Number of groups    : {groups.nunique()}")
print(f"Excluded columns    : {drop_columns}")

X.head()

## 6. Train-Test Split by Event

The holdout split is performed at event level. Complete events are assigned either to the training set or to the test set. This creates a stricter validation scenario than a random row-level split and better reflects the question of how well the model generalizes to unseen measurement events.

In [ ]:
X_train, X_test, y_train, y_test, groups_train, groups_test = split_by_group(
    X,
    y,
    groups,
    test_size=TEST_SIZE,
    random_state=RANDOM_STATE,
    verbose=True,
)

split_distribution = summarize_split_distribution(
    X_train,
    X_test,
    y_train,
    y_test,
    land_use_column=LAND_USE_COLUMN,
    verbose=True,
)

split_distribution

## 7. Preprocessing Pipeline

The preprocessing pipeline applies consistent transformations to numerical and categorical features. Numerical variables are imputed where necessary, while categorical variables are one-hot encoded.

Feature scaling is not required for XGBoost, because tree-based models are invariant to monotonic transformations of individual feature scales. The pipeline is still useful because it ensures that training, validation, and prediction use identical preprocessing steps.

In [ ]:
numeric_features, categorical_features = get_feature_type_lists(X_train, verbose=True)
preprocessor = build_tabular_preprocessor(numeric_features, categorical_features, verbose=True)


## 8. Baseline Model

A baseline model is included as a reference point. It predicts a constant value based on the training target distribution. This makes it possible to judge whether XGBoost learns meaningful structure beyond a simple central tendency of the target.

In [ ]:
# DummyRegressor expects a feature matrix. A zero-column placeholder is enough.
X_train_dummy = np.zeros((len(y_train), 1))
X_test_dummy = np.zeros((len(y_test), 1))

# baseline_model -> baseline_model_reg = DummyRegressor(strategy="median").fit(X_train_dummy, y_train)
baseline_model = DummyRegressor(strategy="median").fit(X_train_dummy, y_train)

baseline_metrics = regression_metrics(
    y_test, baseline_model.predict(X_test_dummy), prefix="baseline_", verbose=True
)

## 9. Initial XGBoost Model

The first XGBoost model is trained on the original target scale. This model provides an initial performance estimate and serves as a direct comparison against the baseline before target transformation and hyperparameter tuning.

In [ ]:
# XGBRegressor hyperparameters can be tuned later using RandomizedSearchCV or GridSearchCV.
model = XGBRegressor(
    objective="reg:squarederror",
    n_estimators=500,
    learning_rate=0.05,
    max_depth=4,
    subsample=0.8,
    colsample_bytree=0.8,
    reg_alpha=0.0,
    reg_lambda=1.0,
    random_state=RANDOM_STATE,
    n_jobs=N_JOBS,
    device= "cpu",#"cuda",
    # tree_method="hist",
)

# Build the initial pipeline with preprocessing and the XGBoost model.
xgb_initial = Pipeline(
    steps=[
        ("preprocess", preprocessor),
        ("model", model),
    ]
)


xgb_initial.fit(X_train, y_train)

initial_predictions = xgb_initial.predict(X_test)

# Evaluate the initial model performance using regression metrics.
initial_metrics = regression_metrics(
    y_test, initial_predictions, prefix="xgb_initial_", verbose=True
)

## 10. Hyperparameter Tuning with Grouped Cross-Validation

The tuned model uses a shifted `log1p` transformation of the target. This transformation is intended to reduce the effect of skewness while still supporting negative target values through a training-set-based shift.

Hyperparameter tuning is performed with grouped cross-validation. The grouping structure is preserved during validation so that observations from the same `Event` are not split across training and validation folds.

In [ ]:
log_transformer = ShiftedLogTargetTransformer(verbose=True)
y_train_log = log_transformer.fit_transform(y_train, verbose=True)

In [ ]:
cv = build_group_kfold(groups_train, max_splits=5, verbose=True)

# Define hyperparameter search space for XGBoost model tuning.
param_distributions = {
    "model__n_estimators": [300, 500, 800, 1200],
    "model__learning_rate": [0.01, 0.02, 0.03, 0.05, 0.08, 0.10],
    "model__max_depth": [2, 3, 4, 5, 6],
    "model__min_child_weight": [1, 3, 5, 7, 10],
    "model__subsample": [0.6, 0.7, 0.8, 0.9, 1.0],
    "model__colsample_bytree": [0.6, 0.7, 0.8, 0.9, 1.0],
    "model__reg_alpha": [0.0, 0.001, 0.01, 0.1, 1.0],
    "model__reg_lambda": [0.5, 1.0, 2.0, 5.0, 10.0],
    "model__gamma": [0.0, 0.01, 0.1, 0.5, 1.0],
}

# Build the pipeline for hyperparameter search with preprocessing and the XGBoost model.
xgb_for_search = Pipeline(
    steps=[
        ("preprocess", preprocessor),
        ("model", model),
    ]
)

search = RandomizedSearchCV(
    estimator=xgb_for_search,
    param_distributions=param_distributions,
    n_iter=N_ITER_SEARCH,
    scoring="neg_root_mean_squared_error",
    cv=cv,
    random_state=RANDOM_STATE,
    n_jobs=N_JOBS,
    verbose=1,
    return_train_score=True,
)

search.fit(X_train, y_train_log, groups=groups_train)

print("Best CV score on transformed target scale:", search.best_score_)
print("Best parameters:")
for parameter, value in search.best_params_.items():
    print(f"- {parameter}: {value}")


In [ ]:
cv_results = (
    pd.DataFrame(search.cv_results_)
    .sort_values("rank_test_score")
    .loc[:, [
        "rank_test_score",
        "mean_test_score",
        "std_test_score",
        "mean_train_score",
        "std_train_score",
        "params",
    ]]
)

cv_results.head(10)


## 11. Final Model Evaluation

The tuned model predicts on the transformed target scale. Predictions are inverse-transformed back to the original N₂O flux scale before evaluation.

The final evaluation is based on the event-level holdout test set. The reported metrics therefore describe performance on events that were not used during model fitting or hyperparameter selection.

In [ ]:
best_xgb_log_model = search.best_estimator_

test_predictions_log = best_xgb_log_model.predict(X_test)

test_predictions = log_transformer.inverse_transform(test_predictions_log, verbose=True)


final_metrics = regression_metrics(
    y_test,
    test_predictions,
    prefix="xgb_tuned_log_",
    verbose=True,
)

# Remove model-specific prefixes so all metric dictionaries share the same index
baseline_metrics_clean = {
    key.replace("baseline_", ""): value
    for key, value in baseline_metrics.items()
}

initial_metrics_clean = {
    key.replace("xgb_initial_", ""): value
    for key, value in initial_metrics.items()
}

final_metrics_clean = {
    key.replace("xgb_tuned_log_", ""): value
    for key, value in final_metrics.items()
}

# Build a clean comparison table with metrics as rows and models as columns
metrics_table = pd.DataFrame(
    {
        "baseline_median": baseline_metrics_clean,
        "xgb_initial_raw": initial_metrics_clean,
        "xgb_tuned_shifted_log": final_metrics_clean,
    }
)

# Keep metrics in a clear and consistent order
metrics_table = metrics_table.loc[
    [
        "mae",
        "rmse",
        "r2",
        "median_ae",
        "bias_mean_residual",
    ]
]

metrics_table

In [ ]:
predictions_df = X_test[[LAND_USE_COLUMN]].copy()
predictions_df[GROUP_COLUMN] = groups_test.values
predictions_df["y_true"] = y_test.values
predictions_df["y_pred"] = test_predictions
predictions_df["residual"] = predictions_df["y_true"] - predictions_df["y_pred"]
predictions_df["absolute_error"] = predictions_df["residual"].abs()

predictions_df.head()


## 12. Error Analysis

This section examines whether prediction errors vary systematically across land-use classes, events, and target-intensity ranges. The goal is to identify model weaknesses that are not visible from global metrics alone.

In [ ]:
land_use_error = metrics_by_group(
    predictions_df,
    group_column=LAND_USE_COLUMN,
    min_count=1,
    verbose=True,
)

land_use_error


In [ ]:
event_error = metrics_by_group(
    predictions_df,
    group_column=GROUP_COLUMN,
    min_count=5,
    verbose=True,
)

event_error.head(20)


### Error Analysis Across Target Quantiles

To evaluate whether the model performs differently for low and high N₂O flux values, the true target values are divided into four quantile-based groups.

Unlike fixed-width bins, quantile bins contain approximately the same number of observations. This makes the error metrics easier to compare across different target ranges, especially because the N₂O flux distribution is strongly skewed.

The grouped metrics show whether prediction errors increase for higher emission values and whether the model performs consistently across the full target distribution.

In [ ]:
# Create four target-value groups with approximately equal numbers of observations.
predictions_df["target_quantile_bin"] = pd.qcut(
    predictions_df["y_true"],
    q=4,
    duplicates="drop",
)

# Display the number of observations in each quantile bin.
print(
    predictions_df["target_quantile_bin"]
    .value_counts()
    .sort_index()
)

# Calculate regression metrics separately for each target-value range.
quantile_error = metrics_by_group(
    predictions_df,
    group_column="target_quantile_bin",
    min_count=1,
    verbose=True,
)

quantile_error

## 13. Feature Importance

Feature importance is extracted from the fitted XGBoost model after preprocessing. One-hot encoded categorical variables are represented by their expanded feature names.

The importance values provide an initial indication of which predictors contribute most strongly to the fitted model. They are interpreted as model-specific importance scores and not as causal effects.

In [ ]:
from src.n2o_ml.xgboost_io import extract_xgb_importance

importance_df = extract_xgb_importance(
    best_xgb_log_model,
    numeric_features=numeric_features,
    categorical_features=categorical_features,
    importance_type="gain",
    verbose=True,
)

importance_df.head(30)


In [ ]:
top_n = 25

plt.figure(figsize=(10, max(5, top_n * 0.28)))
sns.barplot(data= importance_df.head(top_n).sort_values("importance", ascending=False), x="importance", y="feature")
plt.xlabel("Importance (gain)")
plt.ylabel("Feature")
plt.title(f"Top {len(importance_df)} XGBoost Feature Importances")
plt.tight_layout()
plt.show()


## 14. Diagnostic Plots

The diagnostic plots visualize prediction quality and residual behavior on the holdout test set. They are used to inspect systematic deviations, heteroscedasticity, and the model behavior for high-emission observations.

In [ ]:
# Plot predicted values against actual values.
y_true_array = np.asarray(y_test, dtype=float)
y_pred_array = np.asarray(test_predictions, dtype=float)

axis_min = min(y_true_array.min(), y_pred_array.min())
axis_max = max(y_true_array.max(), y_pred_array.max())


plt.figure(figsize=(7, 7))
sns.scatterplot(x=y_true_array, y=y_pred_array, alpha=0.7)
plt.plot([axis_min, axis_max], [axis_min, axis_max], linestyle="--")
plt.xlabel("Actual N2O flux [µg/m²/h]")
plt.ylabel("Predicted N2O flux [µg/m²/h]")
plt.title("Tuned XGBoost: Predicted vs Actual N2O Flux")
plt.tight_layout()
plt.show()



In [ ]:
residuals = y_true_array - y_pred_array

plt.figure(figsize=(8, 5))
sns.scatterplot(x=y_pred_array, y=residuals, alpha=0.7)
plt.axhline(0, linestyle="--")
plt.xlabel("Predicted N2O flux [µg/m²/h]")
plt.ylabel("Residual: actual - predicted")
plt.title("Residuals vs Predicted Values")
plt.tight_layout()
plt.show()

plt.figure(figsize=(8, 5))
sns.histplot(residuals, bins=40, kde=True)
plt.xlabel("Residual: actual - predicted")
plt.ylabel("Frequency")
plt.title("Residual Distribution")
plt.tight_layout()
plt.show()




In [ ]:
plt.figure(figsize=(8, 5))
sns.boxplot(
    data=predictions_df,
    x=LAND_USE_COLUMN,
    y="absolute_error",
    showfliers=False,
)
plt.xlabel("Land use")
plt.ylabel("Absolute error [µg/m²/h]")
plt.title("Absolute Prediction Error by Land Use")
plt.xticks(rotation=45, ha="right")
plt.tight_layout()
plt.show()

In [ ]:
# Optional: inspect predictions for the largest absolute errors.
predictions_df.sort_values("absolute_error", ascending=False).head(20)


## 15. Save Model Artifacts

The fitted model and all relevant evaluation artifacts are exported for later reuse. The saved outputs include:

- the fitted preprocessing and XGBoost pipeline,
- the target shift required for inverse transformation,
- feature lists,
- evaluation metrics,
- holdout predictions,
- feature-importance values.

In [ ]:
best_xgb_log_model

In [ ]:
def save_model_artifacts(
    model: Pipeline,
    target_shift: float,
    metrics_table: pd.DataFrame,
    predictions_df: pd.DataFrame,
    importance_df: pd.DataFrame,
    numeric_features: Sequence[str],
    categorical_features: Sequence[str],
    output_dir: Path,
    verbose: bool = False,
) -> Dict[str, Path]:
    """
    Save model and model-building artifacts to disk.

    Parameters
    ----------
    model : Pipeline
        Fitted model pipeline.
    target_shift : float
        Shift used for target transformation.
    metrics_table : pd.DataFrame
        Evaluation metrics table.
    predictions_df : pd.DataFrame
        Holdout prediction table.
    importance_df : pd.DataFrame
        Feature importance table.
    numeric_features : Sequence[str]
        Numeric feature names.
    categorical_features : Sequence[str]
        Categorical feature names.
    output_dir : Path
        Directory where artifacts should be saved.
    verbose : bool, default=False
        If True, print saved file paths.

    Returns
    -------
    Dict[str, Path]
        Dictionary of artifact names and paths.
    """
    output_dir = Path(output_dir)
    output_dir.mkdir(parents=True, exist_ok=True)

    artifact_bundle = {
        "model": model,
        "target_shift": target_shift,
        "target_transform": "shifted_log1p",
        "numeric_features": list(numeric_features),
        "categorical_features": list(categorical_features),
        "target_column": TARGET_COLUMN,
        "group_column": GROUP_COLUMN,
        "date_column": DATE_COLUMN,
        "land_use_column": LAND_USE_COLUMN,
    }

    model_path = output_dir / "xgboost_n2o_model.joblib"
    metrics_path = output_dir / "xgboost_n2o_metrics.csv"
    predictions_path = output_dir / "xgboost_n2o_holdout_predictions.csv"
    importance_path = output_dir / "xgboost_n2o_feature_importance.csv"

    joblib.dump(artifact_bundle, model_path)
    metrics_table.to_csv(metrics_path, index=True)
    predictions_df.to_csv(predictions_path, index=False)
    importance_df.to_csv(importance_path, index=False)

    paths = {
        "model": model_path,
        "metrics": metrics_path,
        "predictions": predictions_path,
        "feature_importance": importance_path,
    }

    if verbose:
        print("Saved artifacts:")
        for name, path in paths.items():
            print(f"- {name}: {path.resolve()}")

    return paths


ARTIFACT_DIR = project_root / "models" / "xgboost_n2o"

saved_paths = save_model_artifacts(
    model=best_xgb_log_model,
    target_shift=log_transformer.shift_,
    metrics_table=metrics_table,
    predictions_df=predictions_df,
    importance_df=importance_df,
    numeric_features=numeric_features,
    categorical_features=categorical_features,
    output_dir=ARTIFACT_DIR,
    verbose=True,
)

saved_paths


## 16. Optional Final Refit on All Data

The previous model represents the honest evaluation model, because it is trained only on the training split and evaluated on unseen events. After the holdout performance has been documented, a production-style model can optionally be refitted on all available observations using the selected hyperparameters.

This optional refit is not used to revise the holdout metrics. It only creates a final model artifact trained on the maximum available amount of data.

In [ ]:
# Optional production-style refit on all available data.
# This model should not be used for holdout evaluation, because it has seen all observations.

REFIT_ON_ALL_DATA = False

if REFIT_ON_ALL_DATA:
    full_target_shift = calculate_log_shift(y, verbose=True)
    y_full_log = transform_target_log1p_shifted(y, shift=full_target_shift, verbose=True)

    final_model_all_data = clone(best_xgb_log_model)
    final_model_all_data.fit(X, y_full_log)

    final_artifact_dir = project_root / "models" / "xgboost_n2o_final_all_data"
    final_artifact_dir.mkdir(parents=True, exist_ok=True)

    final_bundle = {
        "model": final_model_all_data,
        "target_shift": full_target_shift,
        "target_transform": "shifted_log1p",
        "numeric_features": list(numeric_features),
        "categorical_features": list(categorical_features),
        "target_column": TARGET_COLUMN,
        "group_column": GROUP_COLUMN,
        "date_column": DATE_COLUMN,
        "land_use_column": LAND_USE_COLUMN,
    }

    final_model_path = final_artifact_dir / "xgboost_n2o_final_all_data.joblib"
    joblib.dump(final_bundle, final_model_path)

    print(f"Saved final all-data model to: {final_model_path.resolve()}")
else:
    print("Skipped final refit on all data. Set REFIT_ON_ALL_DATA = True to run it.")


## 17. Summary and Next Steps

This section is reserved for the final interpretation after the notebook has been executed.

The modeling summary should document:

1. the baseline performance compared with the initial XGBoost model,
2. the tuned XGBoost performance on unseen events,
3. the effect of the shifted-log target transformation on MAE, RMSE, and residual behavior,
4. the most relevant predictors according to gain-based feature importance,
5. systematic error differences across land-use classes or events,
6. the model behavior for high-emission observations.

Potential next steps include:

- comparison with additional model families such as Random Forest, HistGradientBoosting, and ElasticNet,
- evaluation with leave-one-event-out cross-validation,
- testing a reduced feature set without model-derived variables such as `Transformation S` and `Transformation C`,
- SHAP-based interpretation for a more detailed explanation of model behavior,
- creation of a compact model card for the project README.